In [ ]:
import os
from pathlib import Path

REVIEW_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "REVIEW_CODE_ROOT").is_file()),
    None,
)
if REVIEW_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted review package.")
os.chdir(REVIEW_ROOT)
REVIEW_FREQUENCY = os.environ.get("BFORTPROB_FREQUENCY", "daily")
if REVIEW_FREQUENCY not in {"daily", "weekly"}:
    raise ValueError("BFORTPROB_FREQUENCY must be daily or weekly.")


GARCH(1,1)
+ Student-t innovations
+ parameters fitted once on training data
+ fixed parameters at test time
+ forward simulation of the complete 20-step return path
+ empirical 0.1,...,0.9 quantiles across simulated paths

In [ ]:
import arch
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from arch import arch_model
from scipy.stats import t as student_t


QUANTILE_LEVELS = np.arange(
    0.1,
    1.0,
    0.1
)

N_SIMULATIONS = 1000
SIMULATION_SEED = 42


# ============================================================
# FIT STUDENT-t GARCH(1,1) ON TRAINING SERIES
# ============================================================

def fit_student_t_garch_11_on_train(
    train_series
):
    """
    Fit StandardScaler + Student-t GARCH(1,1)
    ONCE using the training series only.

    Parameters are then frozen during test evaluation.
    """

    train_series = np.asarray(
        train_series,
        dtype=float
    )

    # --------------------------------------------------------
    # 1. TRAIN-ONLY SCALER
    # --------------------------------------------------------

    scaler = StandardScaler()

    train_std = scaler.fit_transform(
        train_series.reshape(-1, 1)
    ).ravel()

    # --------------------------------------------------------
    # 2. STUDENT-t GARCH(1,1)
    # --------------------------------------------------------

    model = arch_model(
        train_std,
        mean="Constant",
        vol="GARCH",
        p=1,
        q=1,
        dist="t",
        rescale=False,
    )

    result = model.fit(
        disp="off"
    )

    return result, scaler

# ============================================================
# FIXED-PARAMETER STUDENT-t GARCH SIMULATION FORECAST
# ============================================================

def forecast_student_t_garch_11_fixed(
    x,
    trained_result,
    scaler,
    horizon=20,
    quantile_levels=QUANTILE_LEVELS,
    n_simulations=N_SIMULATIONS,
    seed=SIMULATION_SEED,
):
    """
    Forecast one test context using Student-t GARCH(1,1)
    parameters estimated on the training set.

    NO refitting occurs on the test context.

    Multi-step predictive quantiles are obtained from
    simulation paths.
    """

    x = np.asarray(
        x,
        dtype=float
    )

    # --------------------------------------------------------
    # 1. STANDARDIZE TEST CONTEXT USING TRAIN SCALER
    # --------------------------------------------------------

    x_std = scaler.transform(
        x.reshape(-1, 1)
    ).ravel()

    # --------------------------------------------------------
    # 2. SAME MODEL STRUCTURE ON CURRENT OBSERVED HISTORY
    # --------------------------------------------------------

    test_model = arch_model(
        x_std,
        mean="Constant",
        vol="GARCH",
        p=1,
        q=1,
        dist="t",
        rescale=False,
    )

    # --------------------------------------------------------
    # 3. FREEZE TRAINING PARAMETERS
    # --------------------------------------------------------

    fixed_result = test_model.fix(
        trained_result.params
    )

    # --------------------------------------------------------
    # 4. STUDENT-t RANDOM GENERATOR
    #
    # arch's Student-t innovations are standardized to
    # unit variance.
    #
    # If T ~ t_nu:
    #
    # Var(T) = nu/(nu-2)
    #
    # therefore multiply by sqrt((nu-2)/nu)
    # to obtain unit-variance shocks.
    # --------------------------------------------------------

    nu = float(
        trained_result.params["nu"]
    )

    if nu <= 2.0:
        raise ValueError(
            f"Invalid Student-t degrees of freedom: {nu}"
        )

    rng_np = np.random.default_rng(
        seed
    )

    standardization = np.sqrt(
        (nu - 2.0) / nu
    )

    def student_t_rng(size):
        return (
            rng_np.standard_t(
                df=nu,
                size=size,
            )
            * standardization
        )

    # --------------------------------------------------------
    # 5. SIMULATE COMPLETE MULTI-STEP PATHS
    # --------------------------------------------------------

    forecast = fixed_result.forecast(
        horizon=horizon,
        method="simulation",
        simulations=n_simulations,
        rng=student_t_rng,
        reindex=False,
    )

    # --------------------------------------------------------
    # Simulation values contain simulated process values.
    #
    # Expected shape:
    # (1, n_simulations, horizon)
    # --------------------------------------------------------

    simulated_std = np.asarray(
        forecast.simulations.values,
        dtype=float,
    )

    if simulated_std.ndim != 3:
        raise ValueError(
            "Unexpected simulation shape: "
            f"{simulated_std.shape}"
        )

    # Only one forecast origin in this call
    simulated_std = simulated_std[-1]

    # Expected:
    # (n_simulations, horizon)
    if simulated_std.shape != (
        n_simulations,
        horizon,
    ):
        raise ValueError(
            "Unexpected simulation path shape: "
            f"{simulated_std.shape}; "
            f"expected "
            f"({n_simulations}, {horizon})"
        )

    # --------------------------------------------------------
    # 6. EMPIRICAL PREDICTIVE QUANTILES
    # --------------------------------------------------------

    quantiles_std = np.quantile(
        simulated_std,
        quantile_levels,
        axis=0,
    ).T

    # Shape:
    # (horizon, 9)

    # --------------------------------------------------------
    # 7. CONDITIONAL-MEAN POINT FORECAST
    #
    # Preserve original GARCH point-forecast convention.
    # --------------------------------------------------------

    mu_std = (
        forecast.mean
        .iloc[-1]
        .to_numpy(dtype=float)
    )

    # --------------------------------------------------------
    # 8. RETURN EVERYTHING TO ORIGINAL LOG-RETURN SCALE
    # --------------------------------------------------------

    scale = float(
        scaler.scale_[0]
    )

    center = float(
        scaler.mean_[0]
    )

    simulated = (
        simulated_std * scale
        + center
    )

    quantiles = (
        quantiles_std * scale
        + center
    )

    mu = (
        mu_std * scale
        + center
    )

    # Simulation-based predictive SD for diagnostics
    sigma = np.std(
        simulated,
        axis=0,
        ddof=1,
    )

    return (
        mu,
        sigma,
        quantiles,
        simulated,
    )

In [ ]:
QUANTILE_LEVELS = np.arange(0.1, 1.0, 0.1)

print(QUANTILE_LEVELS)

def quantile_crps(
    y,
    quantile_forecasts,
    quantile_levels=QUANTILE_LEVELS
):
    """
    y:
        shape (...,)

    quantile_forecasts:
        shape (..., Q)

    returns:
        CRPS approximation with shape (...)
    """

    y = np.asarray(y, dtype=float)
    qf = np.asarray(quantile_forecasts, dtype=float)
    levels = np.asarray(quantile_levels, dtype=float)

    error = y[..., None] - qf

    pinball = np.maximum(
        levels * error,
        (levels - 1.0) * error
    )

    return 2.0 * np.mean(pinball, axis=-1)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import zlib

# ============================================================
# SETTINGS
# ============================================================

SOURCE_DIR = Path(r"BForTFin_log_return/arima_split_datasets/output_daily")
RESULTS_DIR = Path(r"BForTFin_results/standardised/output_daily/garch_t_1_1_standardized_train_test")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

LOOKBACK = 250
HORIZON = 20
P = 1
Q = 1

ASSET_FOLDERS = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

summary_rows = []


# ============================================================
# LOOP OVER ASSET CLASSES
# ============================================================

for folder_name, asset_class in ASSET_FOLDERS.items():

    input_dir = SOURCE_DIR / folder_name
    output_dir = RESULTS_DIR / folder_name

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    if not input_dir.exists():
        print(f"Missing folder: {input_dir}")
        continue

    train_files = sorted(
        input_dir.glob(
            "train_*_no_preprocess_"
            "lookback-250_horizon-20_slide-1.csv"
        )
    )

    print("\n" + "=" * 80)
    print(f"{asset_class}: {len(train_files)} datasets")
    print("=" * 80)

    # ========================================================
    # LOOP OVER DATASETS
    # ========================================================

    for train_csv in train_files:

        suffix = (
            "_no_preprocess_"
            "lookback-250_"
            "horizon-20_"
            "slide-1.csv"
        )

        dataset_name = (
            train_csv.name
            .removeprefix("train_")
            .removesuffix(suffix)
        )

        # Keep Tether excluded
        if "tether" in dataset_name.lower():
            print(f"Skipping Tether: {dataset_name}")
            continue

        test_csv = input_dir / (
            f"test_{dataset_name}"
            f"_no_preprocess_"
            f"lookback-250_"
            f"horizon-20_"
            f"slide-1.csv"
        )

        if not test_csv.exists():
            print(f"Missing test file: {test_csv}")
            continue

        print("\n" + "-" * 70)
        print(f"Dataset: {dataset_name}")
        print(f"Asset class: {asset_class}")
        print("-" * 70)

        # ====================================================
        # LOAD TRAIN + TEST
        # ====================================================

        train_df = pd.read_csv(train_csv)
        test_df = pd.read_csv(test_csv)

        x_cols = [
            f"X_{i}"
            for i in range(LOOKBACK)
        ]

        y_cols = [
            f"Y_{i}"
            for i in range(HORIZON)
        ]

        X_train = train_df[
            x_cols
        ].to_numpy(dtype=float)

        Y_train = train_df[
            y_cols
        ].to_numpy(dtype=float)

        X_test = test_df[
            x_cols
        ].to_numpy(dtype=float)

        Y_test = test_df[
            y_cols
        ].to_numpy(dtype=float)

        # ====================================================
        # BASIC CHECKS
        # ====================================================

        if len(X_train) == 0:
            print("Skipping: empty training set")
            continue

        if len(X_test) == 0:
            print("Skipping: empty test set")
            continue

        if not np.isfinite(X_train).all():
            print("Skipping: non-finite X_train")
            continue

        if not np.isfinite(Y_train).all():
            print("Skipping: non-finite Y_train")
            continue

        if not np.isfinite(X_test).all():
            print("Skipping: non-finite X_test")
            continue

        if not np.isfinite(Y_test).all():
            print("Skipping: non-finite Y_test")
            continue

        # ====================================================
        # RECONSTRUCT CHRONOLOGICAL TRAINING SERIES
        # slide = 1
        # ====================================================

        train_series = list(
            X_train[0]
        )

        for i in range(1, len(X_train)):
            train_series.extend(
                X_train[i, -1:]
            )

        # Include final known training future
        train_series.extend(
            Y_train[-1]
        )

        train_series = np.asarray(
            train_series,
            dtype=float
        )

        print(
            "Train rows:",
            len(X_train)
        )

        print(
            "Test rows:",
            len(X_test)
        )

        print(
            "Reconstructed train series:",
            len(train_series)
        )

        # ====================================================
        # FIT GARCH(1,1) ONCE
        # ====================================================

        try:

            trained_garch, train_scaler = (
                fit_student_t_garch_11_on_train(
                    train_series
                )
            )

        except Exception as e:

            print(
                f"Training failed for "
                f"{dataset_name}: {e}"
            )

            continue

        print(
            "Student-t GARCH training complete."
        )
        
        print(
            trained_garch.params
        )
        
        print(
            "Estimated nu:",
            trained_garch.params.get(
                "nu",
                np.nan
            )
        )

        dataset_seed = (
            SIMULATION_SEED
            + zlib.crc32(
                dataset_name.encode("utf-8")
            )
        ) % (2**32 - 1)

        # ====================================================
        # STORAGE
        # ====================================================

        N = len(X_test)
        H = HORIZON
        NQ = len(QUANTILE_LEVELS)

        garch_test_quantiles = np.full(
            (N, H, NQ),
            np.nan,
            dtype=float
        )

        garch_test_crps = np.full(
            (N, H),
            np.nan,
            dtype=float
        )

        point_forecasts = np.full(
            (N, H),
            np.nan,
            dtype=float
        )

        failed_windows = []

        # ====================================================
        # TEST LOOP
        # ====================================================

        for i in range(N):

            try:

                
                (
                    mu,
                    sigma,
                    quantiles,
                    simulated
                ) = forecast_student_t_garch_11_fixed(
                    x=X_test[i],
                    trained_result=trained_garch,
                    scaler=train_scaler,
                    horizon=H,
                    quantile_levels=QUANTILE_LEVELS,
                    n_simulations=N_SIMULATIONS,
                    seed = (
                        dataset_seed + i
                    ) % (2**32 - 1)
                )
                

                crps_i = quantile_crps(
                    Y_test[i],
                    quantiles,
                    QUANTILE_LEVELS
                )

                if not np.all(
                    np.isfinite(crps_i)
                ):
                    raise RuntimeError(
                        "Non-finite qCRPS"
                    )

                point_forecasts[i] = mu

                garch_test_quantiles[i] = (
                    quantiles
                )

                garch_test_crps[i] = (
                    crps_i
                )

            except Exception as e:

                failed_windows.append({
                    "window": i,
                    "error": str(e)
                })

            if (i + 1) % 100 == 0:
                print(
                    f"{dataset_name}: "
                    f"{i + 1}/{N}"
                )

        # ====================================================
        # qCRPS
        # ====================================================

        crps_by_horizon = np.nanmean(
            garch_test_crps,
            axis=0
        )

        mean_qcrps = np.nanmean(
            crps_by_horizon
        )

        n_failed = len(
            failed_windows
        )

        n_valid = (
            N - n_failed
        )

        print(
            f"Mean qCRPS: "
            f"{mean_qcrps:.8f}"
        )

        print(
            f"Valid windows: "
            f"{n_valid}/{N}"
        )

        # ====================================================
        # FILE PREFIX
        # ====================================================

        clean_name = (
            dataset_name
            .lower()
            .replace(
                " historical data",
                ""
            )
            .replace(
                " ",
                "_"
            )
            .replace(
                ".",
                ""
            )
        )

        prefix = (
            f"{dataset_name}_"
            f"garch_t_1_1_simulated_train_test"
        )

        # ====================================================
        # SAVE QUANTILES
        # ====================================================

        np.savez_compressed(
            output_dir /
            f"test_{dataset_name}"
            "_no_preprocess_lookback-250_horizon-20_slide-1"
            "_garch-t-1-1-simulated_quantiles.npz",

            quantiles=(
                garch_test_quantiles
            ),

            quantile_levels=(
                QUANTILE_LEVELS
            )
        )

        # ====================================================
        # SAVE FULL CRPS ARRAY
        # ====================================================

        np.savez_compressed(
            output_dir /
            f"test_{dataset_name}"
            "_no_preprocess_lookback-250_horizon-20_slide-1"
            "_garch-t-1-1-simulated_crps.npz",

            crps=(
                garch_test_crps
            )
        )

        # ====================================================
        # SAVE qCRPS BY HORIZON
        # ====================================================

        pd.DataFrame({
            "horizon": np.arange(
                1,
                H + 1
            ),
            "mean_qcrps": (
                crps_by_horizon
            )
        }).to_csv(
            output_dir /
            f"{prefix}_crps_by_horizon.csv",
            index=False
        )

        # ====================================================
        # SAVE POINT FORECAST CSV
        # ====================================================

        point_df = test_df.copy()

        for h, col in enumerate(y_cols):
            point_df[col] = (
                point_forecasts[:, h]
            )

        point_output_path = (
            output_dir
            / (
                f"test_{dataset_name}"
                "_no_preprocess_lookback-250_horizon-20_slide-1"
                "_garch-t-1-1-simulated.csv"
            )
        )

        point_df.to_csv(
            point_output_path,
            index=False
        )

        # ====================================================
        # SAVE FAILURES
        # ====================================================

        if failed_windows:

            pd.DataFrame(
                failed_windows
            ).to_csv(
                output_dir /
                f"test_{dataset_name}"
                "_no_preprocess_lookback-250_horizon-20_slide-1"
                "_garch-t-1-1-simulated_failed_windows.csv",
                index=False
            )

        # ====================================================
        # SUMMARY
        # ====================================================

        summary_rows.append({
            "asset_class": asset_class,
            "dataset": dataset_name,
            "model": "Student-t GARCH(1,1) Simulation",
        
            "mean_qcrps": mean_qcrps,
        
            "nu": float(
                trained_garch.params["nu"]
            ),
        
            "n_simulations": N_SIMULATIONS,
        
            "n_train_windows": len(X_train),
            "n_test_windows": N,
            "n_valid": n_valid,
            "n_failed": n_failed,
            "failure_rate": n_failed / N,
        })


# ============================================================
# SAVE SUMMARY FOR ALL DATASETS
# ============================================================

garch_summary = pd.DataFrame(
    summary_rows
)

summary_path = (
    RESULTS_DIR /
    "garch-t-1-1-simulated_train_test_"
    "summary_all_datasets.csv"
)

garch_summary.to_csv(
    summary_path,
    index=False
)

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print(
    f"Summary saved to:\n"
    f"{summary_path}"
)

display(
    garch_summary
)

In [ ]:
print("Student-t parameters:")
print(trained_garch.params)

print(
    "Simulation shape:",
    simulated.shape
)

print(
    "Quantile shape:",
    quantiles.shape
)

print(
    "Quantile ordering valid:",
    np.all(
        np.diff(
            quantiles,
            axis=1
        ) >= 0
    )
)

print(
    "First horizon quantiles:",
    quantiles[0]
)

print(
    "20th horizon quantiles:",
    quantiles[-1]
)

In [ ]:
print(
    "Any non-finite simulations:",
    ~np.isfinite(simulated).all()
)

# Weekly

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import zlib

# ============================================================
# SETTINGS
# ============================================================

SOURCE_DIR = Path(r"BForTFin_log_return/arima_split_datasets/output_weekly")
RESULTS_DIR = Path(r"BForTFin_results/standardised/output_weekly/garch_t_1_1_standardized_train_test")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

LOOKBACK = 250
HORIZON = 20
P = 1
Q = 1

ASSET_FOLDERS = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

summary_rows = []


# ============================================================
# LOOP OVER ASSET CLASSES
# ============================================================

for folder_name, asset_class in ASSET_FOLDERS.items():

    input_dir = SOURCE_DIR / folder_name
    output_dir = RESULTS_DIR / folder_name

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    if not input_dir.exists():
        print(f"Missing folder: {input_dir}")
        continue

    train_files = sorted(
        input_dir.glob(
            "train_*_no_preprocess_"
            "lookback-250_horizon-20_slide-1.csv"
        )
    )

    print("\n" + "=" * 80)
    print(f"{asset_class}: {len(train_files)} datasets")
    print("=" * 80)

    # ========================================================
    # LOOP OVER DATASETS
    # ========================================================

    for train_csv in train_files:

        suffix = (
            "_no_preprocess_"
            "lookback-250_"
            "horizon-20_"
            "slide-1.csv"
        )

        dataset_name = (
            train_csv.name
            .removeprefix("train_")
            .removesuffix(suffix)
        )

        # Keep Tether excluded
        if "tether" in dataset_name.lower():
            print(f"Skipping Tether: {dataset_name}")
            continue

        test_csv = input_dir / (
            f"test_{dataset_name}"
            f"_no_preprocess_"
            f"lookback-250_"
            f"horizon-20_"
            f"slide-1.csv"
        )

        if not test_csv.exists():
            print(f"Missing test file: {test_csv}")
            continue

        print("\n" + "-" * 70)
        print(f"Dataset: {dataset_name}")
        print(f"Asset class: {asset_class}")
        print("-" * 70)

        # ====================================================
        # LOAD TRAIN + TEST
        # ====================================================

        train_df = pd.read_csv(train_csv)
        test_df = pd.read_csv(test_csv)

        x_cols = [
            f"X_{i}"
            for i in range(LOOKBACK)
        ]

        y_cols = [
            f"Y_{i}"
            for i in range(HORIZON)
        ]

        X_train = train_df[
            x_cols
        ].to_numpy(dtype=float)

        Y_train = train_df[
            y_cols
        ].to_numpy(dtype=float)

        X_test = test_df[
            x_cols
        ].to_numpy(dtype=float)

        Y_test = test_df[
            y_cols
        ].to_numpy(dtype=float)

        # ====================================================
        # BASIC CHECKS
        # ====================================================

        if len(X_train) == 0:
            print("Skipping: empty training set")
            continue

        if len(X_test) == 0:
            print("Skipping: empty test set")
            continue

        if not np.isfinite(X_train).all():
            print("Skipping: non-finite X_train")
            continue

        if not np.isfinite(Y_train).all():
            print("Skipping: non-finite Y_train")
            continue

        if not np.isfinite(X_test).all():
            print("Skipping: non-finite X_test")
            continue

        if not np.isfinite(Y_test).all():
            print("Skipping: non-finite Y_test")
            continue

        # ====================================================
        # RECONSTRUCT CHRONOLOGICAL TRAINING SERIES
        # slide = 1
        # ====================================================

        train_series = list(
            X_train[0]
        )

        for i in range(1, len(X_train)):
            train_series.extend(
                X_train[i, -1:]
            )

        # Include final known training future
        train_series.extend(
            Y_train[-1]
        )

        train_series = np.asarray(
            train_series,
            dtype=float
        )

        print(
            "Train rows:",
            len(X_train)
        )

        print(
            "Test rows:",
            len(X_test)
        )

        print(
            "Reconstructed train series:",
            len(train_series)
        )

        # ====================================================
        # FIT GARCH(1,1) ONCE
        # ====================================================

        try:

            trained_garch, train_scaler = (
                fit_student_t_garch_11_on_train(
                    train_series
                )
            )

        except Exception as e:

            print(
                f"Training failed for "
                f"{dataset_name}: {e}"
            )

            continue

        print(
            "Student-t GARCH training complete."
        )
        
        print(
            trained_garch.params
        )
        
        print(
            "Estimated nu:",
            trained_garch.params.get(
                "nu",
                np.nan
            )
        )

        dataset_seed = (
            SIMULATION_SEED
            + zlib.crc32(
                dataset_name.encode("utf-8")
            )
        ) % (2**32 - 1)

        # ====================================================
        # STORAGE
        # ====================================================

        N = len(X_test)
        H = HORIZON
        NQ = len(QUANTILE_LEVELS)

        garch_test_quantiles = np.full(
            (N, H, NQ),
            np.nan,
            dtype=float
        )

        garch_test_crps = np.full(
            (N, H),
            np.nan,
            dtype=float
        )

        point_forecasts = np.full(
            (N, H),
            np.nan,
            dtype=float
        )

        failed_windows = []

        # ====================================================
        # TEST LOOP
        # ====================================================

        for i in range(N):

            try:

                
                (
                    mu,
                    sigma,
                    quantiles,
                    simulated
                ) = forecast_student_t_garch_11_fixed(
                    x=X_test[i],
                    trained_result=trained_garch,
                    scaler=train_scaler,
                    horizon=H,
                    quantile_levels=QUANTILE_LEVELS,
                    n_simulations=N_SIMULATIONS,
                    seed = (
                        dataset_seed + i
                    ) % (2**32 - 1)
                )
                

                crps_i = quantile_crps(
                    Y_test[i],
                    quantiles,
                    QUANTILE_LEVELS
                )

                if not np.all(
                    np.isfinite(crps_i)
                ):
                    raise RuntimeError(
                        "Non-finite qCRPS"
                    )

                point_forecasts[i] = mu

                garch_test_quantiles[i] = (
                    quantiles
                )

                garch_test_crps[i] = (
                    crps_i
                )

            except Exception as e:

                failed_windows.append({
                    "window": i,
                    "error": str(e)
                })

            if (i + 1) % 100 == 0:
                print(
                    f"{dataset_name}: "
                    f"{i + 1}/{N}"
                )

        # ====================================================
        # qCRPS
        # ====================================================

        crps_by_horizon = np.nanmean(
            garch_test_crps,
            axis=0
        )

        mean_qcrps = np.nanmean(
            crps_by_horizon
        )

        n_failed = len(
            failed_windows
        )

        n_valid = (
            N - n_failed
        )

        print(
            f"Mean qCRPS: "
            f"{mean_qcrps:.8f}"
        )

        print(
            f"Valid windows: "
            f"{n_valid}/{N}"
        )

        # ====================================================
        # FILE PREFIX
        # ====================================================

        clean_name = (
            dataset_name
            .lower()
            .replace(
                " historical data",
                ""
            )
            .replace(
                " ",
                "_"
            )
            .replace(
                ".",
                ""
            )
        )

        prefix = (
            f"{dataset_name}_"
            f"garch_t_1_1_simulated_train_test"
        )

        # ====================================================
        # SAVE QUANTILES
        # ====================================================

        np.savez_compressed(
            output_dir /
            f"test_{dataset_name}"
            "_no_preprocess_lookback-250_horizon-20_slide-1"
            "_garch-t-1-1-simulated_quantiles.npz",

            quantiles=(
                garch_test_quantiles
            ),

            quantile_levels=(
                QUANTILE_LEVELS
            )
        )

        # ====================================================
        # SAVE FULL CRPS ARRAY
        # ====================================================

        np.savez_compressed(
            output_dir /
            f"test_{dataset_name}"
            "_no_preprocess_lookback-250_horizon-20_slide-1"
            "_garch-t-1-1-simulated_crps.npz",

            crps=(
                garch_test_crps
            )
        )

        # ====================================================
        # SAVE qCRPS BY HORIZON
        # ====================================================

        pd.DataFrame({
            "horizon": np.arange(
                1,
                H + 1
            ),
            "mean_qcrps": (
                crps_by_horizon
            )
        }).to_csv(
            output_dir /
            f"{prefix}_crps_by_horizon.csv",
            index=False
        )

        # ====================================================
        # SAVE POINT FORECAST CSV
        # ====================================================

        point_df = test_df.copy()

        for h, col in enumerate(y_cols):
            point_df[col] = (
                point_forecasts[:, h]
            )

        point_output_path = (
            output_dir
            / (
                f"test_{dataset_name}"
                "_no_preprocess_lookback-250_horizon-20_slide-1"
                "_garch-t-1-1-simulated.csv"
            )
        )

        point_df.to_csv(
            point_output_path,
            index=False
        )

        # ====================================================
        # SAVE FAILURES
        # ====================================================

        if failed_windows:

            pd.DataFrame(
                failed_windows
            ).to_csv(
                output_dir /
                f"test_{dataset_name}"
                "_no_preprocess_lookback-250_horizon-20_slide-1"
                "_garch-t-1-1-simulated_failed_windows.csv",
                index=False
            )

        # ====================================================
        # SUMMARY
        # ====================================================

        summary_rows.append({
            "asset_class": asset_class,
            "dataset": dataset_name,
            "model": "Student-t GARCH(1,1) Simulation",
        
            "mean_qcrps": mean_qcrps,
        
            "nu": float(
                trained_garch.params["nu"]
            ),
        
            "n_simulations": N_SIMULATIONS,
        
            "n_train_windows": len(X_train),
            "n_test_windows": N,
            "n_valid": n_valid,
            "n_failed": n_failed,
            "failure_rate": n_failed / N,
        })


# ============================================================
# SAVE SUMMARY FOR ALL DATASETS
# ============================================================

garch_summary = pd.DataFrame(
    summary_rows
)

summary_path = (
    RESULTS_DIR /
    "garch-t-1-1-simulated_train_test_"
    "summary_all_datasets.csv"
)

garch_summary.to_csv(
    summary_path,
    index=False
)

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)

print(
    f"Summary saved to:\n"
    f"{summary_path}"
)

display(
    garch_summary
)

# Evaluation
1. compare Student-\(t\) GARCH vs original Gaussian GARCH on qCRPS;
2. compute QSS for Student-\(t\) GARCH using the exact same rolling empirical q0.1 reference as before.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================

ROOT = Path(
    r"."
    r"\BForTFin_results\standardised"
)

FREQUENCY = "daily"   # change to "weekly" for weekly

BASE = ROOT / f"output_{FREQUENCY}"

GAUSSIAN_SUMMARY = (
    BASE
    / "garch_standardized_train_test"
    / "garch_1_1_train_test_summary_all_datasets.csv"
)

STUDENT_T_SUMMARY = (
    BASE
    / "garch_t_1_1_standardized_train_test"
    / "garch-t-1-1-simulated_train_test_summary_all_datasets.csv"
)

OUT_DIR = (
    BASE
    / "garch_t_vs_gaussian_evaluation"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# LOAD
# ============================================================

gaussian = pd.read_csv(
    GAUSSIAN_SUMMARY
)

student = pd.read_csv(
    STUDENT_T_SUMMARY
)

print("Gaussian rows:", len(gaussian))
print("Student-t rows:", len(student))

# ============================================================
# STANDARDISE ONLY COLUMN NAMES
# ============================================================

gaussian = gaussian.rename(columns={
    "mean_qcrps": "gaussian_qcrps"
})

student = student.rename(columns={
    "mean_qcrps": "student_t_qcrps"
})

# Keep useful Student-t diagnostics if present
student_keep = [
    c for c in [
        "asset_class",
        "dataset",
        "student_t_qcrps",
        "nu",
        "n_simulations",
        "n_train_windows",
        "n_test_windows",
        "n_valid",
        "n_failed",
        "failure_rate",
    ]
    if c in student.columns
]

student = student[
    student_keep
].copy()

gaussian = gaussian[
    [
        "asset_class",
        "dataset",
        "gaussian_qcrps",
    ]
].copy()

# ============================================================
# MERGE
# ============================================================

comparison = gaussian.merge(
    student,
    on=[
        "asset_class",
        "dataset",
    ],
    how="inner",
)

print(
    "Matched datasets:",
    len(comparison)
)

# ============================================================
# IMPROVEMENT
# Lower qCRPS is better
# ============================================================

comparison[
    "absolute_qcrps_improvement"
] = (
    comparison["gaussian_qcrps"]
    - comparison["student_t_qcrps"]
)

comparison[
    "relative_qcrps_improvement_pct"
] = (
    100.0
    * comparison["absolute_qcrps_improvement"]
    / comparison["gaussian_qcrps"]
)

comparison[
    "student_t_better"
] = (
    comparison["student_t_qcrps"]
    < comparison["gaussian_qcrps"]
)

# ============================================================
# OVERALL SUMMARY
# ============================================================

overall = pd.DataFrame([{
    "frequency": FREQUENCY,

    "n_datasets":
        len(comparison),

    "mean_gaussian_qcrps":
        comparison[
            "gaussian_qcrps"
        ].mean(),

    "mean_student_t_qcrps":
        comparison[
            "student_t_qcrps"
        ].mean(),

    "mean_relative_improvement_pct":
        comparison[
            "relative_qcrps_improvement_pct"
        ].mean(),

    "median_relative_improvement_pct":
        comparison[
            "relative_qcrps_improvement_pct"
        ].median(),

    "fraction_student_t_better":
        comparison[
            "student_t_better"
        ].mean(),

    "n_student_t_better":
        comparison[
            "student_t_better"
        ].sum(),
}])

overall[
    "fraction_student_t_better"
] *= 100.0

# ============================================================
# BY ASSET CLASS
# ============================================================

by_asset = (
    comparison
    .groupby(
        "asset_class",
        as_index=False
    )
    .agg(
        n_datasets=(
            "dataset",
            "count"
        ),
        mean_gaussian_qcrps=(
            "gaussian_qcrps",
            "mean"
        ),
        mean_student_t_qcrps=(
            "student_t_qcrps",
            "mean"
        ),
        mean_relative_improvement_pct=(
            "relative_qcrps_improvement_pct",
            "mean"
        ),
        median_relative_improvement_pct=(
            "relative_qcrps_improvement_pct",
            "median"
        ),
        fraction_student_t_better=(
            "student_t_better",
            "mean"
        ),
    )
)

by_asset[
    "fraction_student_t_better"
] *= 100.0

# ============================================================
# OPTIONAL nu SUMMARY
# ============================================================

if "nu" in comparison.columns:

    nu_summary = (
        comparison["nu"]
        .describe(
            percentiles=[
                0.10,
                0.25,
                0.50,
                0.75,
                0.90,
            ]
        )
    )

    print("\nStudent-t nu summary:")
    print(nu_summary)

# ============================================================
# SAVE + PRINT
# ============================================================

comparison.to_csv(
    OUT_DIR
    / "student_t_vs_gaussian_qcrps_by_dataset.csv",
    index=False,
)

overall.to_csv(
    OUT_DIR
    / "student_t_vs_gaussian_qcrps_overall.csv",
    index=False,
)

by_asset.to_csv(
    OUT_DIR
    / "student_t_vs_gaussian_qcrps_by_asset_class.csv",
    index=False,
)

print("\nOVERALL:")
print(
    overall.to_string(
        index=False
    )
)

print("\nBY ASSET CLASS:")
print(
    by_asset.to_string(
        index=False
    )
)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================

ROOT = Path(
    r"."
    r"\BForTFin_results\standardised"
)

FREQUENCY = "weekly"   # change to "weekly" for weekly

BASE = ROOT / f"output_{FREQUENCY}"

GAUSSIAN_SUMMARY = (
    BASE
    / "garch_standardized_train_test"
    / "garch_1_1_train_test_summary_all_datasets.csv"
)

STUDENT_T_SUMMARY = (
    BASE
    / "garch_t_1_1_standardized_train_test"
    / "garch-t-1-1-simulated_train_test_summary_all_datasets.csv"
)

OUT_DIR = (
    BASE
    / "garch_t_vs_gaussian_evaluation"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ============================================================
# LOAD
# ============================================================

gaussian = pd.read_csv(
    GAUSSIAN_SUMMARY
)

student = pd.read_csv(
    STUDENT_T_SUMMARY
)

print("Gaussian rows:", len(gaussian))
print("Student-t rows:", len(student))

# ============================================================
# STANDARDISE ONLY COLUMN NAMES
# ============================================================

gaussian = gaussian.rename(columns={
    "mean_qcrps": "gaussian_qcrps"
})

student = student.rename(columns={
    "mean_qcrps": "student_t_qcrps"
})

# Keep useful Student-t diagnostics if present
student_keep = [
    c for c in [
        "asset_class",
        "dataset",
        "student_t_qcrps",
        "nu",
        "n_simulations",
        "n_train_windows",
        "n_test_windows",
        "n_valid",
        "n_failed",
        "failure_rate",
    ]
    if c in student.columns
]

student = student[
    student_keep
].copy()

gaussian = gaussian[
    [
        "asset_class",
        "dataset",
        "gaussian_qcrps",
    ]
].copy()

# ============================================================
# MERGE
# ============================================================

comparison = gaussian.merge(
    student,
    on=[
        "asset_class",
        "dataset",
    ],
    how="inner",
)

print(
    "Matched datasets:",
    len(comparison)
)

# ============================================================
# IMPROVEMENT
# Lower qCRPS is better
# ============================================================

comparison[
    "absolute_qcrps_improvement"
] = (
    comparison["gaussian_qcrps"]
    - comparison["student_t_qcrps"]
)

comparison[
    "relative_qcrps_improvement_pct"
] = (
    100.0
    * comparison["absolute_qcrps_improvement"]
    / comparison["gaussian_qcrps"]
)

comparison[
    "student_t_better"
] = (
    comparison["student_t_qcrps"]
    < comparison["gaussian_qcrps"]
)

# ============================================================
# OVERALL SUMMARY
# ============================================================

overall = pd.DataFrame([{
    "frequency": FREQUENCY,

    "n_datasets":
        len(comparison),

    "mean_gaussian_qcrps":
        comparison[
            "gaussian_qcrps"
        ].mean(),

    "mean_student_t_qcrps":
        comparison[
            "student_t_qcrps"
        ].mean(),

    "mean_relative_improvement_pct":
        comparison[
            "relative_qcrps_improvement_pct"
        ].mean(),

    "median_relative_improvement_pct":
        comparison[
            "relative_qcrps_improvement_pct"
        ].median(),

    "fraction_student_t_better":
        comparison[
            "student_t_better"
        ].mean(),

    "n_student_t_better":
        comparison[
            "student_t_better"
        ].sum(),
}])

overall[
    "fraction_student_t_better"
] *= 100.0

# ============================================================
# BY ASSET CLASS
# ============================================================

by_asset = (
    comparison
    .groupby(
        "asset_class",
        as_index=False
    )
    .agg(
        n_datasets=(
            "dataset",
            "count"
        ),
        mean_gaussian_qcrps=(
            "gaussian_qcrps",
            "mean"
        ),
        mean_student_t_qcrps=(
            "student_t_qcrps",
            "mean"
        ),
        mean_relative_improvement_pct=(
            "relative_qcrps_improvement_pct",
            "mean"
        ),
        median_relative_improvement_pct=(
            "relative_qcrps_improvement_pct",
            "median"
        ),
        fraction_student_t_better=(
            "student_t_better",
            "mean"
        ),
    )
)

by_asset[
    "fraction_student_t_better"
] *= 100.0

# ============================================================
# OPTIONAL nu SUMMARY
# ============================================================

if "nu" in comparison.columns:

    nu_summary = (
        comparison["nu"]
        .describe(
            percentiles=[
                0.10,
                0.25,
                0.50,
                0.75,
                0.90,
            ]
        )
    )

    print("\nStudent-t nu summary:")
    print(nu_summary)

# ============================================================
# SAVE + PRINT
# ============================================================

comparison.to_csv(
    OUT_DIR
    / "student_t_vs_gaussian_qcrps_by_dataset.csv",
    index=False,
)

overall.to_csv(
    OUT_DIR
    / "student_t_vs_gaussian_qcrps_overall.csv",
    index=False,
)

by_asset.to_csv(
    OUT_DIR
    / "student_t_vs_gaussian_qcrps_by_asset_class.csv",
    index=False,
)

print("\nOVERALL:")
print(
    overall.to_string(
        index=False
    )
)

print("\nBY ASSET CLASS:")
print(
    by_asset.to_string(
        index=False
    )
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

TAU = 0.1
LOOKBACK = 250
HORIZON = 20

ROOT = Path(
    r"."
    r"\BForTFin_results\standardised"
)

FREQUENCY = "daily"   # change to weekly

BASE = ROOT / f"output_{FREQUENCY}"

PROJECT_ROOT = Path(
    r"."
)

SPLIT_ROOT = (
    PROJECT_ROOT
    / "BForTFin_log_return"
    / "arima_split_datasets"
    / f"output_{FREQUENCY}"
)

RESULT_ROOT = (
    PROJECT_ROOT
    / "BForTFin_results"
    / "standardised"
    / f"output_{FREQUENCY}"
    / "garch_t_1_1_standardized_train_test"
)

OUT_DIR = (
    PROJECT_ROOT
    / "BForTFin_results"
    / "standardised"
    / f"output_{FREQUENCY}"
    / "garch_t_vs_gaussian_evaluation"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ASSET_FOLDERS = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

X_COLS = [
    f"X_{i}"
    for i in range(LOOKBACK)
]

Y_COLS = [
    f"Y_{i}"
    for i in range(HORIZON)
]


# ============================================================
# QUANTILE LOSS
# ============================================================

def quantile_loss(
    y,
    q,
    tau=TAU,
):
    error = y - q

    return np.maximum(
        tau * error,
        (tau - 1.0) * error
    )


# ============================================================
# FIND STUDENT-t QUANTILE FILE
# ============================================================

def find_student_t_quantile_file(
    folder_name,
    dataset_name,
):

    folder = (
        RESULT_ROOT
        / folder_name
    )

    # Try exact naming first
    exact = (
        folder
        / (
            f"test_{dataset_name}"
            "_no_preprocess_lookback-250_horizon-20_slide-1"
            "_garch-t-1-1-simulated_quantiles.npz"
        )
    )

    if exact.exists():
        return exact

    # Fallback in case suffix differs slightly
    matches = list(
        folder.glob(
            f"{dataset_name}*quantiles.npz"
        )
    )

    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"Could not uniquely find quantile file for "
        f"{dataset_name}; found {len(matches)}"
    )


# ============================================================
# LOOP
# ============================================================

rows = []

for folder_name, asset_class in ASSET_FOLDERS.items():

    split_dir = (
        SPLIT_ROOT
        / folder_name
    )

    test_files = sorted(
        split_dir.glob(
            "test_*_no_preprocess_"
            "lookback-250_horizon-20_slide-1.csv"
        )
    )

    for test_path in test_files:

        dataset_name = (
            test_path.name
            .removeprefix("test_")
            .removesuffix(
                "_no_preprocess_"
                "lookback-250_"
                "horizon-20_"
                "slide-1.csv"
            )
        )

        # Weekly Solana excluded
        if (
            FREQUENCY == "weekly"
            and "solana" in dataset_name.lower()
        ):
            continue

        test_df = pd.read_csv(
            test_path
        )

        X = test_df[
            X_COLS
        ].to_numpy(dtype=float)

        Y = test_df[
            Y_COLS
        ].to_numpy(dtype=float)

        quantile_path = (
            find_student_t_quantile_file(
                folder_name,
                dataset_name,
            )
        )

        npz = np.load(
            quantile_path
        )

        quantiles = np.asarray(
            npz["quantiles"],
            dtype=float,
        )

        levels = np.asarray(
            npz["quantile_levels"],
            dtype=float,
        )

        # ---------------------------------------------
        # CHECK SHAPE
        # expected (N, 20, 9)
        # ---------------------------------------------

        if quantiles.shape != (
            len(test_df),
            HORIZON,
            len(levels),
        ):
            raise ValueError(
                f"{dataset_name}: unexpected quantile shape "
                f"{quantiles.shape}"
            )

        # ---------------------------------------------
        # FIND q0.1 INDEX
        # ---------------------------------------------

        tau_idx = np.where(
            np.isclose(
                levels,
                TAU
            )
        )[0]

        if len(tau_idx) != 1:
            raise ValueError(
                f"{dataset_name}: q0.1 not found uniquely"
            )

        tau_idx = int(
            tau_idx[0]
        )

        model_q = (
            quantiles[
                :,
                :,
                tau_idx
            ]
        )

        # ---------------------------------------------
        # EMPIRICAL q0.1 REFERENCE
        #
        # one empirical q0.1 from each 250-point
        # context, used across all 20 horizons
        # ---------------------------------------------

        empirical_q = np.quantile(
            X,
            TAU,
            axis=1,
        )

        empirical_q = (
            empirical_q[:, None]
        )

        empirical_q = np.repeat(
            empirical_q,
            HORIZON,
            axis=1,
        )

        # ---------------------------------------------
        # PINBALL LOSS
        # ---------------------------------------------

        model_loss = quantile_loss(
            Y,
            model_q,
            TAU,
        )

        ref_loss = quantile_loss(
            Y,
            empirical_q,
            TAU,
        )

        mean_model_loss = float(
            np.mean(
                model_loss
            )
        )

        mean_ref_loss = float(
            np.mean(
                ref_loss
            )
        )

        qss = (
            1.0
            - (
                mean_model_loss
                / mean_ref_loss
            )
        )

        # ---------------------------------------------
        # BY HORIZON
        # ---------------------------------------------

        model_loss_h = np.mean(
            model_loss,
            axis=0,
        )

        ref_loss_h = np.mean(
            ref_loss,
            axis=0,
        )

        qss_h = (
            1.0
            - (
                model_loss_h
                / ref_loss_h
            )
        )

        rows.append({
            "asset_class": asset_class,
            "dataset": dataset_name,
            "model": "Student-t GARCH(1,1) Simulation",
            "qss_0.1": qss,
            "mean_model_ql_0.1": mean_model_loss,
            "mean_reference_ql_0.1": mean_ref_loss,
            "qss_h1": qss_h[0],
            "qss_h5": qss_h[4],
            "qss_h20": qss_h[19],
        })


# ============================================================
# SUMMARY
# ============================================================

qss_df = pd.DataFrame(
    rows
)

qss_df.to_csv(
    OUT_DIR
    / "student_t_garch_qss_by_dataset.csv",
    index=False,
)

summary = pd.DataFrame([{
    "frequency": FREQUENCY,
    "n_datasets": len(qss_df),
    "mean_qss": qss_df["qss_0.1"].mean(),
    "median_qss": qss_df["qss_0.1"].median(),
    "fraction_positive_qss":
        100.0 * (
            qss_df["qss_0.1"] > 0
        ).mean(),
}])

summary.to_csv(
    OUT_DIR
    / "student_t_garch_qss_summary.csv",
    index=False,
)

by_asset = (
    qss_df
    .groupby(
        "asset_class",
        as_index=False
    )
    .agg(
        n_datasets=(
            "dataset",
            "count"
        ),
        mean_qss=(
            "qss_0.1",
            "mean"
        ),
        median_qss=(
            "qss_0.1",
            "median"
        ),
        fraction_positive_qss=(
            "qss_0.1",
            lambda x:
                100.0 * (
                    x > 0
                ).mean()
        ),
    )
)

by_asset.to_csv(
    OUT_DIR
    / "student_t_garch_qss_by_asset_class.csv",
    index=False,
)

print("\nQSS SUMMARY:")
print(
    summary.to_string(
        index=False
    )
)

print("\nBY ASSET CLASS:")
print(
    by_asset.to_string(
        index=False
    )
)

existing_qss = pd.read_csv(
    BASE
    / "qss_summary_all_models.csv"
)

gaussian_qss = (
    existing_qss[
        existing_qss["model"]
        .astype(str)
        .str.contains(
            "garch",
            case=False,
            na=False
        )
    ][
        [
            "asset_class",
            "dataset",
            "qss_0.1",
        ]
    ]
    .copy()
)

def canonical_dataset_name(name):
    name = str(name)

    suffixes = [
        "_no_preprocess_lookback-250_horizon-20_slide-1",
        "_lookback-250_horizon-20_slide-1",
    ]

    for suffix in suffixes:
        if name.endswith(suffix):
            name = name[:-len(suffix)]

    return name.strip()


gaussian_qss["dataset"] = (
    gaussian_qss["dataset"]
    .map(canonical_dataset_name)
)

student_qss = (
    qss_df[
        [
            "asset_class",
            "dataset",
            "qss_0.1",
        ]
    ]
    .copy()
)

student_qss["dataset"] = (
    student_qss["dataset"]
    .map(canonical_dataset_name)
)

gaussian_qss = gaussian_qss.rename(
    columns={
        "qss_0.1": "gaussian_qss"
    }
)

student_qss = student_qss.rename(
    columns={
        "qss_0.1": "student_t_qss"
    }
)

print("Gaussian rows:", len(gaussian_qss))
print("Student-t rows:", len(student_qss))

print(
    "Gaussian unique datasets:",
    gaussian_qss["dataset"].nunique()
)

print(
    "Student-t unique datasets:",
    student_qss["dataset"].nunique()
)

qss_comparison = gaussian_qss.merge(
    student_qss,
    on=[
        "asset_class",
        "dataset",
    ],
    how="inner",
    validate="one_to_one",
)

print(
    "Matched datasets:",
    len(qss_comparison)
)



qss_comparison[
    "student_t_improvement"
] = (
    qss_comparison["student_t_qss"]
    - qss_comparison["gaussian_qss"]
)

qss_comparison[
    "student_t_better"
] = (
    qss_comparison["student_t_qss"]
    > qss_comparison["gaussian_qss"]
)

print("\nGaussian vs Student-t QSS:")
print(
    qss_comparison[
        [
            "gaussian_qss",
            "student_t_qss",
            "student_t_improvement",
        ]
    ].describe()
)

print(
    "\nStudent-t better on:",
    f"{100 * qss_comparison['student_t_better'].mean():.1f}%",
    "of datasets"
)

qss_comparison.to_csv(
    OUT_DIR
    / "student_t_vs_gaussian_qss_by_dataset.csv",
    index=False,
)

print(
    qss_df.sort_values("qss_0.1")[
        [
            "asset_class",
            "dataset",
            "qss_0.1",
            "mean_model_ql_0.1",
            "mean_reference_ql_0.1",
        ]
    ].head(15).to_string(index=False)
)

In [ ]:
print("Models in existing QSS:")
print(existing_qss["model"].unique())

print("\nGaussian-looking models:")
print(
    existing_qss.loc[
        existing_qss["model"]
        .astype(str)
        .str.contains("garch", case=False, na=False),
        ["model", "asset_class", "dataset", "qss_0.1"]
    ].head(20)
)

print("\nStudent-t sample names:")
print(
    qss_df[
        ["asset_class", "dataset"]
    ].head(20)
)

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

TAU = 0.1
LOOKBACK = 250
HORIZON = 20

ROOT = Path(
    r"."
    r"\BForTFin_results\standardised"
)

FREQUENCY = "weekly"   # change to weekly

BASE = ROOT / f"output_{FREQUENCY}"

PROJECT_ROOT = Path(
    r"."
)

SPLIT_ROOT = (
    PROJECT_ROOT
    / "BForTFin_log_return"
    / "arima_split_datasets"
    / f"output_{FREQUENCY}"
)

RESULT_ROOT = (
    PROJECT_ROOT
    / "BForTFin_results"
    / "standardised"
    / f"output_{FREQUENCY}"
    / "garch_t_1_1_standardized_train_test"
)

OUT_DIR = (
    PROJECT_ROOT
    / "BForTFin_results"
    / "standardised"
    / f"output_{FREQUENCY}"
    / "garch_t_vs_gaussian_evaluation"
)

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

ASSET_FOLDERS = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

X_COLS = [
    f"X_{i}"
    for i in range(LOOKBACK)
]

Y_COLS = [
    f"Y_{i}"
    for i in range(HORIZON)
]


# ============================================================
# QUANTILE LOSS
# ============================================================

def quantile_loss(
    y,
    q,
    tau=TAU,
):
    error = y - q

    return np.maximum(
        tau * error,
        (tau - 1.0) * error
    )


# ============================================================
# FIND STUDENT-t QUANTILE FILE
# ============================================================

def find_student_t_quantile_file(
    folder_name,
    dataset_name,
):

    folder = (
        RESULT_ROOT
        / folder_name
    )

    # Try exact naming first
    exact = (
        folder
        / (
            f"test_{dataset_name}"
            "_no_preprocess_lookback-250_horizon-20_slide-1"
            "_garch-t-1-1-simulated_quantiles.npz"
        )
    )

    if exact.exists():
        return exact

    # Fallback in case suffix differs slightly
    matches = list(
        folder.glob(
            f"{dataset_name}*quantiles.npz"
        )
    )

    if len(matches) == 1:
        return matches[0]

    raise FileNotFoundError(
        f"Could not uniquely find quantile file for "
        f"{dataset_name}; found {len(matches)}"
    )


# ============================================================
# LOOP
# ============================================================

rows = []

for folder_name, asset_class in ASSET_FOLDERS.items():

    split_dir = (
        SPLIT_ROOT
        / folder_name
    )

    test_files = sorted(
        split_dir.glob(
            "test_*_no_preprocess_"
            "lookback-250_horizon-20_slide-1.csv"
        )
    )

    for test_path in test_files:

        dataset_name = (
            test_path.name
            .removeprefix("test_")
            .removesuffix(
                "_no_preprocess_"
                "lookback-250_"
                "horizon-20_"
                "slide-1.csv"
            )
        )

        # Weekly Solana excluded
        if (
            FREQUENCY == "weekly"
            and "solana" in dataset_name.lower()
        ):
            continue

        test_df = pd.read_csv(
            test_path
        )

        X = test_df[
            X_COLS
        ].to_numpy(dtype=float)

        Y = test_df[
            Y_COLS
        ].to_numpy(dtype=float)

        quantile_path = (
            find_student_t_quantile_file(
                folder_name,
                dataset_name,
            )
        )

        npz = np.load(
            quantile_path
        )

        quantiles = np.asarray(
            npz["quantiles"],
            dtype=float,
        )

        levels = np.asarray(
            npz["quantile_levels"],
            dtype=float,
        )

        # ---------------------------------------------
        # CHECK SHAPE
        # expected (N, 20, 9)
        # ---------------------------------------------

        if quantiles.shape != (
            len(test_df),
            HORIZON,
            len(levels),
        ):
            raise ValueError(
                f"{dataset_name}: unexpected quantile shape "
                f"{quantiles.shape}"
            )

        # ---------------------------------------------
        # FIND q0.1 INDEX
        # ---------------------------------------------

        tau_idx = np.where(
            np.isclose(
                levels,
                TAU
            )
        )[0]

        if len(tau_idx) != 1:
            raise ValueError(
                f"{dataset_name}: q0.1 not found uniquely"
            )

        tau_idx = int(
            tau_idx[0]
        )

        model_q = (
            quantiles[
                :,
                :,
                tau_idx
            ]
        )

        # ---------------------------------------------
        # EMPIRICAL q0.1 REFERENCE
        #
        # one empirical q0.1 from each 250-point
        # context, used across all 20 horizons
        # ---------------------------------------------

        empirical_q = np.quantile(
            X,
            TAU,
            axis=1,
        )

        empirical_q = (
            empirical_q[:, None]
        )

        empirical_q = np.repeat(
            empirical_q,
            HORIZON,
            axis=1,
        )

        # ---------------------------------------------
        # PINBALL LOSS
        # ---------------------------------------------

        model_loss = quantile_loss(
            Y,
            model_q,
            TAU,
        )

        ref_loss = quantile_loss(
            Y,
            empirical_q,
            TAU,
        )

        mean_model_loss = float(
            np.mean(
                model_loss
            )
        )

        mean_ref_loss = float(
            np.mean(
                ref_loss
            )
        )

        qss = (
            1.0
            - (
                mean_model_loss
                / mean_ref_loss
            )
        )

        # ---------------------------------------------
        # BY HORIZON
        # ---------------------------------------------

        model_loss_h = np.mean(
            model_loss,
            axis=0,
        )

        ref_loss_h = np.mean(
            ref_loss,
            axis=0,
        )

        qss_h = (
            1.0
            - (
                model_loss_h
                / ref_loss_h
            )
        )

        rows.append({
            "asset_class": asset_class,
            "dataset": dataset_name,
            "model": "Student-t GARCH(1,1) Simulation",
            "qss_0.1": qss,
            "mean_model_ql_0.1": mean_model_loss,
            "mean_reference_ql_0.1": mean_ref_loss,
            "qss_h1": qss_h[0],
            "qss_h5": qss_h[4],
            "qss_h20": qss_h[19],
        })


# ============================================================
# SUMMARY
# ============================================================

qss_df = pd.DataFrame(
    rows
)

qss_df.to_csv(
    OUT_DIR
    / "student_t_garch_qss_by_dataset.csv",
    index=False,
)

summary = pd.DataFrame([{
    "frequency": FREQUENCY,
    "n_datasets": len(qss_df),
    "mean_qss": qss_df["qss_0.1"].mean(),
    "median_qss": qss_df["qss_0.1"].median(),
    "fraction_positive_qss":
        100.0 * (
            qss_df["qss_0.1"] > 0
        ).mean(),
}])

summary.to_csv(
    OUT_DIR
    / "student_t_garch_qss_summary.csv",
    index=False,
)

by_asset = (
    qss_df
    .groupby(
        "asset_class",
        as_index=False
    )
    .agg(
        n_datasets=(
            "dataset",
            "count"
        ),
        mean_qss=(
            "qss_0.1",
            "mean"
        ),
        median_qss=(
            "qss_0.1",
            "median"
        ),
        fraction_positive_qss=(
            "qss_0.1",
            lambda x:
                100.0 * (
                    x > 0
                ).mean()
        ),
    )
)

by_asset.to_csv(
    OUT_DIR
    / "student_t_garch_qss_by_asset_class.csv",
    index=False,
)

print("\nQSS SUMMARY:")
print(
    summary.to_string(
        index=False
    )
)

print("\nBY ASSET CLASS:")
print(
    by_asset.to_string(
        index=False
    )
)

existing_qss = pd.read_csv(
    BASE
    / "qss_summary_all_models.csv"
)

gaussian_qss = (
    existing_qss[
        existing_qss["model"]
        .astype(str)
        .str.contains(
            "garch",
            case=False,
            na=False
        )
    ][
        [
            "asset_class",
            "dataset",
            "qss_0.1",
        ]
    ]
    .copy()
)

def canonical_dataset_name(name):
    name = str(name)

    suffixes = [
        "_no_preprocess_lookback-250_horizon-20_slide-1",
        "_lookback-250_horizon-20_slide-1",
    ]

    for suffix in suffixes:
        if name.endswith(suffix):
            name = name[:-len(suffix)]

    return name.strip()


gaussian_qss["dataset"] = (
    gaussian_qss["dataset"]
    .map(canonical_dataset_name)
)

student_qss = (
    qss_df[
        [
            "asset_class",
            "dataset",
            "qss_0.1",
        ]
    ]
    .copy()
)

student_qss["dataset"] = (
    student_qss["dataset"]
    .map(canonical_dataset_name)
)

gaussian_qss = gaussian_qss.rename(
    columns={
        "qss_0.1": "gaussian_qss"
    }
)

student_qss = student_qss.rename(
    columns={
        "qss_0.1": "student_t_qss"
    }
)

print("Gaussian rows:", len(gaussian_qss))
print("Student-t rows:", len(student_qss))

print(
    "Gaussian unique datasets:",
    gaussian_qss["dataset"].nunique()
)

print(
    "Student-t unique datasets:",
    student_qss["dataset"].nunique()
)

qss_comparison = gaussian_qss.merge(
    student_qss,
    on=[
        "asset_class",
        "dataset",
    ],
    how="inner",
    validate="one_to_one",
)

print(
    "Matched datasets:",
    len(qss_comparison)
)



qss_comparison[
    "student_t_improvement"
] = (
    qss_comparison["student_t_qss"]
    - qss_comparison["gaussian_qss"]
)

qss_comparison[
    "student_t_better"
] = (
    qss_comparison["student_t_qss"]
    > qss_comparison["gaussian_qss"]
)

print("\nGaussian vs Student-t QSS:")
print(
    qss_comparison[
        [
            "gaussian_qss",
            "student_t_qss",
            "student_t_improvement",
        ]
    ].describe()
)

print(
    "\nStudent-t better on:",
    f"{100 * qss_comparison['student_t_better'].mean():.1f}%",
    "of datasets"
)

qss_comparison.to_csv(
    OUT_DIR
    / "student_t_vs_gaussian_qss_by_dataset.csv",
    index=False,
)

print(
    qss_df.sort_values("qss_0.1")[
        [
            "asset_class",
            "dataset",
            "qss_0.1",
            "mean_model_ql_0.1",
            "mean_reference_ql_0.1",
        ]
    ].head(15).to_string(index=False)
)

# Ranking

Rank models within each dataset, then average ranks across datasets. Lower rank is better.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================

FREQUENCY = "daily"   # change to "weekly"

ROOT = Path(
    r"."
    r"\BForTFin_results\standardised"
)

BASE = ROOT / f"output_{FREQUENCY}"

OUT_DIR = BASE / "student_t_garch_ranking_update"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# HELPERS
# ============================================================

def canonical_dataset_name(name):
    name = str(name).strip()

    suffixes = [
        "_no_preprocess_lookback-250_horizon-20_slide-1",
        "_lookback-250_horizon-20_slide-1",
    ]

    for suffix in suffixes:
        if name.endswith(suffix):
            name = name[:-len(suffix)]

    return name.strip()


def print_ranking(title, df):
    print(f"\n{title}")
    print(
        df[
            [
                "model",
                "mean_rank",
                "rank"
            ]
        ].to_string(index=False)
    )


# ============================================================
# PART 1 — qCRPS
# ============================================================

# Existing all-model probabilistic results
qcrps_old = pd.read_csv(
    BASE / "all_models_crps_by_dataset.csv"
)

# Student-t GARCH summary
student_summary = pd.read_csv(
    BASE
    / "garch_t_1_1_standardized_train_test"
    / "garch-t-1-1-simulated_train_test_summary_all_datasets.csv"
)

# Standardise names
qcrps_old["dataset"] = (
    qcrps_old["dataset"]
    .map(canonical_dataset_name)
)

student_summary["dataset"] = (
    student_summary["dataset"]
    .map(canonical_dataset_name)
)

# Student-t rows in same format as existing file
student_qcrps = student_summary[
    [
        "asset_class",
        "dataset",
        "mean_qcrps"
    ]
].copy()

student_qcrps["model"] = (
    "Student-t GARCH"
)

student_qcrps = student_qcrps[
    [
        "asset_class",
        "dataset",
        "model",
        "mean_qcrps"
    ]
]

# Remove any accidental existing Student-t row
qcrps_old = qcrps_old[
    ~qcrps_old["model"]
    .astype(str)
    .str.contains(
        "student",
        case=False,
        na=False
    )
].copy()

# Combine
qcrps_all = pd.concat(
    [
        qcrps_old,
        student_qcrps
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

print("\nqCRPS dataset counts by model:")
print(
    qcrps_all
    .groupby("model")["dataset"]
    .nunique()
    .sort_values()
)

# ------------------------------------------------------------
# Rank WITHIN dataset
# lower qCRPS = better
# ------------------------------------------------------------

qcrps_all["dataset_rank"] = (
    qcrps_all
    .groupby(
        [
            "asset_class",
            "dataset"
        ]
    )["mean_qcrps"]
    .rank(
        method="average",
        ascending=True
    )
)

# Overall mean rank
qcrps_ranking = (
    qcrps_all
    .groupby(
        "model",
        as_index=False
    )
    .agg(
        mean_rank=(
            "dataset_rank",
            "mean"
        ),
        mean_qcrps=(
            "mean_qcrps",
            "mean"
        ),
        n_datasets=(
            "dataset",
            "nunique"
        ),
    )
    .sort_values(
        "mean_rank"
    )
    .reset_index(drop=True)
)

qcrps_ranking["rank"] = (
    np.arange(
        1,
        len(qcrps_ranking) + 1
    )
)

print_ranking(
    f"{FREQUENCY.upper()} qCRPS RANKING",
    qcrps_ranking
)


# ============================================================
# PART 2 — QSS
# ============================================================

qss_old = pd.read_csv(
    BASE / "qss_summary_all_models.csv"
)

student_qss = pd.read_csv(
    BASE
    / "garch_t_vs_gaussian_evaluation"
    / "student_t_garch_qss_by_dataset.csv"
)

qss_old["dataset"] = (
    qss_old["dataset"]
    .map(canonical_dataset_name)
)

student_qss["dataset"] = (
    student_qss["dataset"]
    .map(canonical_dataset_name)
)

student_qss["model"] = (
    "Student-t GARCH"
)

student_qss = student_qss[
    [
        "asset_class",
        "dataset",
        "model",
        "qss_0.1"
    ]
]

# Remove accidental duplicate if rerunning
qss_old = qss_old[
    ~qss_old["model"]
    .astype(str)
    .str.contains(
        "student",
        case=False,
        na=False
    )
].copy()

qss_all = pd.concat(
    [
        qss_old,
        student_qss
    ],
    ignore_index=True
)

print("\nQSS dataset counts by model:")
print(
    qss_all
    .groupby("model")["dataset"]
    .nunique()
    .sort_values()
)

# ------------------------------------------------------------
# Rank WITHIN dataset
# higher QSS = better
# ------------------------------------------------------------

qss_all["dataset_rank"] = (
    qss_all
    .groupby(
        [
            "asset_class",
            "dataset"
        ]
    )["qss_0.1"]
    .rank(
        method="average",
        ascending=False
    )
)

qss_ranking = (
    qss_all
    .groupby(
        "model",
        as_index=False
    )
    .agg(
        mean_rank=(
            "dataset_rank",
            "mean"
        ),
        mean_qss=(
            "qss_0.1",
            "mean"
        ),
        median_qss=(
            "qss_0.1",
            "median"
        ),
        n_datasets=(
            "dataset",
            "nunique"
        ),
    )
    .sort_values(
        "mean_rank"
    )
    .reset_index(drop=True)
)

qss_ranking["rank"] = (
    np.arange(
        1,
        len(qss_ranking) + 1
    )
)

print_ranking(
    f"{FREQUENCY.upper()} QSS RANKING",
    qss_ranking
)


# ============================================================
# PART 3 — EQUAL ASSET-CLASS WEIGHTING
# ============================================================

def equal_asset_class_ranking(
    df,
    metric,
    ascending,
):

    temp = df.copy()

    temp["dataset_rank"] = (
        temp
        .groupby(
            [
                "asset_class",
                "dataset"
            ]
        )[metric]
        .rank(
            method="average",
            ascending=ascending
        )
    )

    # First average ranks within each asset class
    class_ranks = (
        temp
        .groupby(
            [
                "model",
                "asset_class"
            ],
            as_index=False
        )
        .agg(
            asset_class_mean_rank=(
                "dataset_rank",
                "mean"
            )
        )
    )

    # Then equal-weight asset classes
    overall = (
        class_ranks
        .groupby(
            "model",
            as_index=False
        )
        .agg(
            equal_asset_mean_rank=(
                "asset_class_mean_rank",
                "mean"
            )
        )
        .sort_values(
            "equal_asset_mean_rank"
        )
        .reset_index(drop=True)
    )

    overall["rank"] = (
        np.arange(
            1,
            len(overall) + 1
        )
    )

    return overall


qcrps_equal = equal_asset_class_ranking(
    qcrps_all,
    metric="mean_qcrps",
    ascending=True
)

qss_equal = equal_asset_class_ranking(
    qss_all,
    metric="qss_0.1",
    ascending=False
)

print("\nqCRPS — equal asset-class weighting")
print(
    qcrps_equal.to_string(
        index=False
    )
)

print("\nQSS — equal asset-class weighting")
print(
    qss_equal.to_string(
        index=False
    )
)


# ============================================================
# SAVE EVERYTHING
# ============================================================

qcrps_all.to_csv(
    OUT_DIR
    / "all_models_qcrps_with_student_t_garch.csv",
    index=False
)

qcrps_ranking.to_csv(
    OUT_DIR
    / "qcrps_ranking_with_student_t_garch.csv",
    index=False
)

qcrps_equal.to_csv(
    OUT_DIR
    / "qcrps_equal_asset_weight_ranking_with_student_t_garch.csv",
    index=False
)

qss_all.to_csv(
    OUT_DIR
    / "all_models_qss_with_student_t_garch.csv",
    index=False
)

qss_ranking.to_csv(
    OUT_DIR
    / "qss_ranking_with_student_t_garch.csv",
    index=False
)

qss_equal.to_csv(
    OUT_DIR
    / "qss_equal_asset_weight_ranking_with_student_t_garch.csv",
    index=False
)

print(
    "\nSaved ranking outputs to:",
    OUT_DIR
)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# CONFIG
# ============================================================

FREQUENCY = "weekly"   # change to "weekly"

ROOT = Path(
    r"."
    r"\BForTFin_results\standardised"
)

BASE = ROOT / f"output_{FREQUENCY}"

OUT_DIR = BASE / "student_t_garch_ranking_update"
OUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# HELPERS
# ============================================================

def canonical_dataset_name(name):
    name = str(name).strip()

    suffixes = [
        "_no_preprocess_lookback-250_horizon-20_slide-1",
        "_lookback-250_horizon-20_slide-1",
    ]

    for suffix in suffixes:
        if name.endswith(suffix):
            name = name[:-len(suffix)]

    return name.strip()


def print_ranking(title, df):
    print(f"\n{title}")
    print(
        df[
            [
                "model",
                "mean_rank",
                "rank"
            ]
        ].to_string(index=False)
    )


# ============================================================
# PART 1 — qCRPS
# ============================================================

# Existing all-model probabilistic results
qcrps_old = pd.read_csv(
    BASE / "all_models_crps_by_dataset.csv"
)

# Student-t GARCH summary
student_summary = pd.read_csv(
    BASE
    / "garch_t_1_1_standardized_train_test"
    / "garch-t-1-1-simulated_train_test_summary_all_datasets.csv"
)

# Standardise names
qcrps_old["dataset"] = (
    qcrps_old["dataset"]
    .map(canonical_dataset_name)
)

student_summary["dataset"] = (
    student_summary["dataset"]
    .map(canonical_dataset_name)
)

# Student-t rows in same format as existing file
student_qcrps = student_summary[
    [
        "asset_class",
        "dataset",
        "mean_qcrps"
    ]
].copy()

student_qcrps["model"] = (
    "Student-t GARCH"
)

student_qcrps = student_qcrps[
    [
        "asset_class",
        "dataset",
        "model",
        "mean_qcrps"
    ]
]

# Remove any accidental existing Student-t row
qcrps_old = qcrps_old[
    ~qcrps_old["model"]
    .astype(str)
    .str.contains(
        "student",
        case=False,
        na=False
    )
].copy()

# Combine
qcrps_all = pd.concat(
    [
        qcrps_old,
        student_qcrps
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# Sanity checks
# ------------------------------------------------------------

print("\nqCRPS dataset counts by model:")
print(
    qcrps_all
    .groupby("model")["dataset"]
    .nunique()
    .sort_values()
)

# ------------------------------------------------------------
# Rank WITHIN dataset
# lower qCRPS = better
# ------------------------------------------------------------

qcrps_all["dataset_rank"] = (
    qcrps_all
    .groupby(
        [
            "asset_class",
            "dataset"
        ]
    )["mean_qcrps"]
    .rank(
        method="average",
        ascending=True
    )
)

# Overall mean rank
qcrps_ranking = (
    qcrps_all
    .groupby(
        "model",
        as_index=False
    )
    .agg(
        mean_rank=(
            "dataset_rank",
            "mean"
        ),
        mean_qcrps=(
            "mean_qcrps",
            "mean"
        ),
        n_datasets=(
            "dataset",
            "nunique"
        ),
    )
    .sort_values(
        "mean_rank"
    )
    .reset_index(drop=True)
)

qcrps_ranking["rank"] = (
    np.arange(
        1,
        len(qcrps_ranking) + 1
    )
)

print_ranking(
    f"{FREQUENCY.upper()} qCRPS RANKING",
    qcrps_ranking
)


# ============================================================
# PART 2 — QSS
# ============================================================

qss_old = pd.read_csv(
    BASE / "qss_summary_all_models.csv"
)

student_qss = pd.read_csv(
    BASE
    / "garch_t_vs_gaussian_evaluation"
    / "student_t_garch_qss_by_dataset.csv"
)

qss_old["dataset"] = (
    qss_old["dataset"]
    .map(canonical_dataset_name)
)

student_qss["dataset"] = (
    student_qss["dataset"]
    .map(canonical_dataset_name)
)

student_qss["model"] = (
    "Student-t GARCH"
)

student_qss = student_qss[
    [
        "asset_class",
        "dataset",
        "model",
        "qss_0.1"
    ]
]

# Remove accidental duplicate if rerunning
qss_old = qss_old[
    ~qss_old["model"]
    .astype(str)
    .str.contains(
        "student",
        case=False,
        na=False
    )
].copy()

qss_all = pd.concat(
    [
        qss_old,
        student_qss
    ],
    ignore_index=True
)

print("\nQSS dataset counts by model:")
print(
    qss_all
    .groupby("model")["dataset"]
    .nunique()
    .sort_values()
)

# ------------------------------------------------------------
# Rank WITHIN dataset
# higher QSS = better
# ------------------------------------------------------------

qss_all["dataset_rank"] = (
    qss_all
    .groupby(
        [
            "asset_class",
            "dataset"
        ]
    )["qss_0.1"]
    .rank(
        method="average",
        ascending=False
    )
)

qss_ranking = (
    qss_all
    .groupby(
        "model",
        as_index=False
    )
    .agg(
        mean_rank=(
            "dataset_rank",
            "mean"
        ),
        mean_qss=(
            "qss_0.1",
            "mean"
        ),
        median_qss=(
            "qss_0.1",
            "median"
        ),
        n_datasets=(
            "dataset",
            "nunique"
        ),
    )
    .sort_values(
        "mean_rank"
    )
    .reset_index(drop=True)
)

qss_ranking["rank"] = (
    np.arange(
        1,
        len(qss_ranking) + 1
    )
)

print_ranking(
    f"{FREQUENCY.upper()} QSS RANKING",
    qss_ranking
)


# ============================================================
# PART 3 — EQUAL ASSET-CLASS WEIGHTING
# ============================================================

def equal_asset_class_ranking(
    df,
    metric,
    ascending,
):

    temp = df.copy()

    temp["dataset_rank"] = (
        temp
        .groupby(
            [
                "asset_class",
                "dataset"
            ]
        )[metric]
        .rank(
            method="average",
            ascending=ascending
        )
    )

    # First average ranks within each asset class
    class_ranks = (
        temp
        .groupby(
            [
                "model",
                "asset_class"
            ],
            as_index=False
        )
        .agg(
            asset_class_mean_rank=(
                "dataset_rank",
                "mean"
            )
        )
    )

    # Then equal-weight asset classes
    overall = (
        class_ranks
        .groupby(
            "model",
            as_index=False
        )
        .agg(
            equal_asset_mean_rank=(
                "asset_class_mean_rank",
                "mean"
            )
        )
        .sort_values(
            "equal_asset_mean_rank"
        )
        .reset_index(drop=True)
    )

    overall["rank"] = (
        np.arange(
            1,
            len(overall) + 1
        )
    )

    return overall


qcrps_equal = equal_asset_class_ranking(
    qcrps_all,
    metric="mean_qcrps",
    ascending=True
)

qss_equal = equal_asset_class_ranking(
    qss_all,
    metric="qss_0.1",
    ascending=False
)

print("\nqCRPS — equal asset-class weighting")
print(
    qcrps_equal.to_string(
        index=False
    )
)

print("\nQSS — equal asset-class weighting")
print(
    qss_equal.to_string(
        index=False
    )
)


# ============================================================
# SAVE EVERYTHING
# ============================================================

qcrps_all.to_csv(
    OUT_DIR
    / "all_models_qcrps_with_student_t_garch.csv",
    index=False
)

qcrps_ranking.to_csv(
    OUT_DIR
    / "qcrps_ranking_with_student_t_garch.csv",
    index=False
)

qcrps_equal.to_csv(
    OUT_DIR
    / "qcrps_equal_asset_weight_ranking_with_student_t_garch.csv",
    index=False
)

qss_all.to_csv(
    OUT_DIR
    / "all_models_qss_with_student_t_garch.csv",
    index=False
)

qss_ranking.to_csv(
    OUT_DIR
    / "qss_ranking_with_student_t_garch.csv",
    index=False
)

qss_equal.to_csv(
    OUT_DIR
    / "qss_equal_asset_weight_ranking_with_student_t_garch.csv",
    index=False
)

print(
    "\nSaved ranking outputs to:",
    OUT_DIR
)

# Statistical significance

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 20,
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 18,
    "figure.titlesize": 20,
})

### Daily CRPS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


def significance_analysis(csv_path, metric_col, output_prefix):
    # ------------------------------------------------------------
    # 1. Load data
    # ------------------------------------------------------------
    df = pd.read_csv(csv_path)

    print(f"\n=== {output_prefix.upper()} ===")
    print(f"Datasets: {df['dataset'].nunique()}")
    print(f"Models: {df['model'].nunique()}")
    print(df['model'].unique())

    # ------------------------------------------------------------
    # 2. Dataset × Model matrix
    # ------------------------------------------------------------
    pivot = df.pivot(
        index='dataset',
        columns='model',
        values=metric_col
    )

    # Friedman requires every dataset to contain every model.
    complete = pivot.dropna().rename(columns={"Chronos-T5": "Chronos" , "GARCH(1,1)": "Gaussian GARCH"})

    print(f"Complete datasets used: {len(complete)}")

    if len(complete) != len(pivot):
        print(
            f"WARNING: dropped {len(pivot) - len(complete)} datasets "
            "because at least one model was missing."
        )

    # ------------------------------------------------------------
    # 3. Rank models within every dataset
    #    1 = lowest metric = best rank
    # ------------------------------------------------------------
    ranks = complete.rank(
        axis=1,
        ascending=True,
        method='average'
    )

    avg_ranks = ranks.mean(axis=0).sort_values()

    print("\nAverage ranks:")
    print(avg_ranks)

    # ------------------------------------------------------------
    # 4. Friedman test
    # ------------------------------------------------------------
    stat, p_value = friedmanchisquare(
        *[complete[col].values for col in complete.columns]
    )

    print(
        f"\nFriedman chi-square = {stat:.4f}, "
        f"p = {p_value:.4e}"
    )

    # ------------------------------------------------------------
    # 5. Nemenyi post-hoc test
    # ------------------------------------------------------------
    if p_value < 0.05:
        p_matrix = sp.posthoc_nemenyi_friedman(complete)

        print("\nNemenyi pairwise p-values:")
        print(p_matrix)

        # Save pairwise p-values
        p_matrix.to_csv(
            f"{output_prefix}_nemenyi_pvalues_updated.csv"
        )

        # --------------------------------------------------------
        # 6. Critical-difference diagram
        # --------------------------------------------------------
        fig, ax = plt.subplots(figsize=(12, 5))

        sp.critical_difference_diagram(
            ranks=avg_ranks,
            sig_matrix=p_matrix,
            ax=ax,
            text_h_margin=0.02
        )

        for line in ax.lines:
            line.set_linewidth(4)
            line.set_markersize(8)
        
        for text in ax.texts:
            text.set_fontsize(17)
            

        plt.title(
            f"{output_prefix.upper()} Critical Difference Diagram Daily\n"
            f"Friedman p = {p_value:.2e}"
        )

        plt.tight_layout()
        plt.savefig(
            f"{output_prefix}_critical_difference_updated.png",
            dpi=300,
            bbox_inches='tight'
        )

        plt.show()
        plt.close()

    else:
        p_matrix = None
        print(
            "\nFriedman test was not significant; "
            "Nemenyi post-hoc comparisons are not interpreted."
        )

    # ------------------------------------------------------------
    # 7. Save ranks
    # ------------------------------------------------------------
    avg_rank_df = (
        avg_ranks
        .rename('average_rank')
        .reset_index()
    )

    avg_rank_df.to_csv(
        f"{output_prefix}_average_ranks_updated.csv",
        index=False
    )

    return complete, ranks, avg_ranks, stat, p_value, p_matrix


# ================================================================
# MASE
# ================================================================
# mase_results = significance_analysis(
#     csv_path="BForTFin_results/standardised/output_weekly/all_models_mase_by_dataset.csv",
#     metric_col="mean_mase",
#     output_prefix="mase"
# )


# ================================================================
# CRPS
# ================================================================
crps_results = significance_analysis(
    csv_path=r"BForTFin_results/standardised/output_daily/student_t_garch_ranking_update/all_models_qcrps_with_student_t_garch.csv",
    metric_col="mean_qcrps",
    output_prefix="crps"
)

### Weekly CRPS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


def significance_analysis(csv_path, metric_col, output_prefix):
    # ------------------------------------------------------------
    # 1. Load data
    # ------------------------------------------------------------
    df = pd.read_csv(csv_path)

    print(f"\n=== {output_prefix.upper()} ===")
    print(f"Datasets: {df['dataset'].nunique()}")
    print(f"Models: {df['model'].nunique()}")
    print(df['model'].unique())

    # ------------------------------------------------------------
    # 2. Dataset × Model matrix
    # ------------------------------------------------------------
    pivot = df.pivot(
        index='dataset',
        columns='model',
        values=metric_col
    )

    # Friedman requires every dataset to contain every model.
    complete = pivot.dropna().rename(columns={"Chronos-T5": "Chronos" , "GARCH(1,1)": "Gaussian GARCH"})

    print(f"Complete datasets used: {len(complete)}")

    if len(complete) != len(pivot):
        print(
            f"WARNING: dropped {len(pivot) - len(complete)} datasets "
            "because at least one model was missing."
        )

    # ------------------------------------------------------------
    # 3. Rank models within every dataset
    #    1 = lowest metric = best rank
    # ------------------------------------------------------------
    ranks = complete.rank(
        axis=1,
        ascending=True,
        method='average'
    )

    avg_ranks = ranks.mean(axis=0).sort_values()

    print("\nAverage ranks:")
    print(avg_ranks)

    # ------------------------------------------------------------
    # 4. Friedman test
    # ------------------------------------------------------------
    stat, p_value = friedmanchisquare(
        *[complete[col].values for col in complete.columns]
    )

    print(
        f"\nFriedman chi-square = {stat:.4f}, "
        f"p = {p_value:.4e}"
    )

    # ------------------------------------------------------------
    # 5. Nemenyi post-hoc test
    # ------------------------------------------------------------
    if p_value < 0.05:
        p_matrix = sp.posthoc_nemenyi_friedman(complete)

        print("\nNemenyi pairwise p-values:")
        print(p_matrix)

        # Save pairwise p-values
        p_matrix.to_csv(
            f"{output_prefix}_nemenyi_pvalues_updated_weekly.csv"
        )

        # --------------------------------------------------------
        # 6. Critical-difference diagram
        # --------------------------------------------------------
        fig, ax = plt.subplots(figsize=(12, 5))

        sp.critical_difference_diagram(
            ranks=avg_ranks,
            sig_matrix=p_matrix,
            ax=ax,
            text_h_margin=0.02
        )

        for line in ax.lines:
            line.set_linewidth(4)
            line.set_markersize(8)
        
        for text in ax.texts:
            text.set_fontsize(17)
            

        plt.title(
            f"{output_prefix.upper()} Critical Difference Diagram Weekly\n"
            f"Friedman p = {p_value:.2e}"
        )

        plt.tight_layout()
        plt.savefig(
            f"{output_prefix}_critical_difference_updated_weekly.png",
            dpi=300,
            bbox_inches='tight'
        )

        plt.show()
        plt.close()

    else:
        p_matrix = None
        print(
            "\nFriedman test was not significant; "
            "Nemenyi post-hoc comparisons are not interpreted."
        )

    # ------------------------------------------------------------
    # 7. Save ranks
    # ------------------------------------------------------------
    avg_rank_df = (
        avg_ranks
        .rename('average_rank')
        .reset_index()
    )

    avg_rank_df.to_csv(
        f"{output_prefix}_average_ranks_updated_weekly.csv",
        index=False
    )

    return complete, ranks, avg_ranks, stat, p_value, p_matrix


# ================================================================
# MASE
# ================================================================
# mase_results = significance_analysis(
#     csv_path="BForTFin_results/standardised/output_weekly/all_models_mase_by_dataset.csv",
#     metric_col="mean_mase",
#     output_prefix="mase"
# )


# ================================================================
# CRPS
# ================================================================
crps_results = significance_analysis(
    csv_path=r"BForTFin_results/standardised/output_weekly/student_t_garch_ranking_update/all_models_qcrps_with_student_t_garch.csv",
    metric_col="mean_qcrps",
    output_prefix="crps"
)

### Daily QSS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


def overall_significance_analysis(
    csv_path,
    metric_col,
    metric_name,
    higher_is_better=False,
    output_prefix=None
):
    df = pd.read_csv(csv_path)

    print("\n" + "=" * 80)
    print(f"OVERALL {metric_name.upper()} SIGNIFICANCE ANALYSIS")
    print("=" * 80)

    print("Datasets:", df["dataset"].nunique())
    print("Models:", df["model"].nunique())
    print("Models:", list(df["model"].unique()))

    # ------------------------------------------------------------
    # 1. Check duplicates
    # ------------------------------------------------------------
    duplicates = df.duplicated(
        subset=["dataset", "model"],
        keep=False
    )

    if duplicates.any():
        print("\nDuplicate dataset/model rows found:")
        print(
            df.loc[
                duplicates,
                ["dataset", "model", metric_col]
            ]
        )
        raise ValueError("Duplicate dataset/model entries detected.")

    # ------------------------------------------------------------
    # 2. Dataset × Model matrix
    # ------------------------------------------------------------
    pivot = df.pivot(
        index="dataset",
        columns="model",
        values=metric_col
    )

    print("\nOriginal matrix shape:", pivot.shape)

    # Friedman needs complete blocks
    complete = pivot.dropna(axis=0, how="any").rename(columns={"Chronos-T5": "Chronos" , "GARCH(1,1)": "Gaussian GARCH"})

    print(
        f"Complete datasets used: {len(complete)} / {len(pivot)}"
    )

    if len(complete) != len(pivot):
        print(
            f"Dropped {len(pivot) - len(complete)} datasets "
            "because at least one model was missing."
        )

    # ------------------------------------------------------------
    # 3. Rank within each dataset
    # ------------------------------------------------------------
    ranks = complete.rank(
        axis=1,
        ascending=not higher_is_better,
        method="average"
    )

    avg_ranks = ranks.mean(axis=0).sort_values()

    print("\nAverage ranks:")
    print(avg_ranks)

    # ------------------------------------------------------------
    # 4. Friedman test
    # ------------------------------------------------------------
    stat, p_value = friedmanchisquare(
        *[
            complete[col].to_numpy()
            for col in complete.columns
        ]
    )

    print("\nFriedman test:")
    print(f"Statistic = {stat:.6f}")
    print(f"p-value   = {p_value:.6e}")

    # ------------------------------------------------------------
    # 5. Nemenyi post-hoc
    # ------------------------------------------------------------
    if p_value < 0.05:

        p_matrix = sp.posthoc_nemenyi_friedman(
            complete
        )

        print("\nNemenyi p-values:")
        print(p_matrix)

        if output_prefix is not None:
            p_matrix.to_csv(
                f"{output_prefix}_nemenyi_pvalues_updated.csv"
            )

        # --------------------------------------------------------
        # 6. Critical-difference diagram
        # --------------------------------------------------------
        fig, ax = plt.subplots(figsize=(12, 5))

        sp.critical_difference_diagram(
            ranks=avg_ranks,
            sig_matrix=p_matrix,
            ax=ax,
            text_h_margin=0.02
        )

        # thicker lines + larger text
        for line in ax.lines:
            line.set_linewidth(4)
            line.set_markersize(8)

        for text in ax.texts:
            text.set_fontsize(17)

        ax.set_title(
            f"Overall {metric_name} Critical Difference Diagram\n"
            f"Friedman p = {p_value:.2e}",
            fontsize=18
        )

        plt.tight_layout()

        if output_prefix is not None:
            plt.savefig(
                f"{output_prefix}_critical_difference_updated.png",
                dpi=300,
                bbox_inches="tight"
            )

        plt.show()
        plt.close()

    else:
        p_matrix = None
        print(
            "\nNo significant overall difference detected."
        )

    # ------------------------------------------------------------
    # 7. Save average ranks
    # ------------------------------------------------------------
    rank_df = (
        avg_ranks
        .rename("average_rank")
        .reset_index()
    )

    rank_df["friedman_statistic"] = stat
    rank_df["friedman_pvalue"] = p_value
    rank_df["n_datasets"] = len(complete)

    if output_prefix is not None:
        rank_df.to_csv(
            f"{output_prefix}_average_ranks_updated.csv",
            index=False
        )

    return {
        "pivot": complete,
        "ranks": ranks,
        "average_ranks": avg_ranks,
        "friedman_statistic": stat,
        "friedman_pvalue": p_value,
        "nemenyi": p_matrix
    }

In [ ]:
#DAILY_DIR = Path(r"BForTFin_results/standardised/output_weekly")

# rmse_results = overall_significance_analysis(
#     csv_path=WEEKLY_DIR / "rmse_summary_all_models.csv",
#     metric_col="rmse",
#     metric_name="RMSE",
#     higher_is_better=False,
#     output_prefix="rmse_overall"
# )

qss_results = overall_significance_analysis(
    csv_path=r"BForTFin_results/standardised/output_daily/student_t_garch_ranking_update/all_models_qss_with_student_t_garch.csv",
    metric_col="qss_0.1",
    metric_name="QSS",
    higher_is_better=True,
    output_prefix="qss_overall"
)

### Weekly QSS

In [ ]:
qss_results = overall_significance_analysis(
    csv_path=r"BForTFin_results/standardised/output_weekly/student_t_garch_ranking_update/all_models_qss_with_student_t_garch.csv",
    metric_col="qss_0.1",
    metric_name="QSS",
    higher_is_better=True,
    output_prefix="qss_overall"
)